# 🗺️ Notebook 6 — Real Use Cases: The Decision Flowchart

> **The vibe:** You've learned all the tools. Now you're the engineer. Given a task, which settings do you choose?  
> **Key word:** MATCH — match the distribution to the task  
> **Time:** ~12 minutes

---

## 🎯 The Core Principle

```
Every task has a "desired distribution."
Your job is to set parameters that produce that distribution.
```

Not the other way around.


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
plt.rcParams.update({'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'axes.edgecolor':'#30363d','text.color':'#e6edf3','axes.labelcolor':'#8b949e',
    'xtick.color':'#8b949e','ytick.color':'#8b949e','grid.color':'#21262d',
    'axes.grid':True,'grid.linewidth':0.5,'font.family':'monospace'})

def softmax(x): e=np.exp(x-x.max()); return e/e.sum()
def apply_temp(l, T): return softmax(l / T)
def apply_top_k(probs, k):
    if k<=0 or k>=len(probs): return probs.copy()
    f=np.zeros_like(probs); top=np.argsort(probs)[-k:]; f[top]=probs[top]; return f/f.sum()
def apply_top_p(probs, p):
    si=np.argsort(probs)[::-1]; cs=np.cumsum(probs[si]); cut=np.searchsorted(cs,p)+1
    f=np.zeros_like(probs); f[si[:cut]]=probs[si[:cut]]; return f/f.sum(), cut
def pipeline(logits, T=1.0, k=0, p=1.0):
    s1=apply_temp(logits,T); s2=apply_top_k(s1,k) if k>0 else s1.copy()
    s3,ns=apply_top_p(s2,p); return s3, ns

# 4 task-shaped distributions
TASKS = {
    'Classification': {
        'logits': np.array([4.5,0.3,0.1,-0.2,-0.5,-1.0,-1.2,-1.5,-2.0,-2.5]),
        'tokens': ['APPROVE','REJECT','REVIEW','FORMAL','CASUAL','ESCALATE','DELAY','AUDIT','HOLD','BLOCK'],
        'T': 0.2, 'k': 0, 'p': 1.0, 'color': '#ff6b9d',
        'description': '🎯 ONE correct answer
Need: near-deterministic',
    },
    'Summarization': {
        'logits': np.array([2.1,1.9,1.7,1.3,0.8,0.5,0.2,-0.2,-0.5,-1.0]),
        'tokens': ['BENEFIT','COST','TIMELINE','STAKEHOLDER','RISK','OPPORTUNITY','CONSTRAINT','DETAIL','TANGENT','NOISE'],
        'T': 0.6, 'k': 0, 'p': 0.95, 'color': '#ff9f43',
        'description': '📝 Several good options
Need: focused + faithful',
    },
    'Code Gen': {
        'logits': np.array([1.2,1.1,1.0,0.9,0.8,0.7,0.6,0.5,0.4,0.3]),
        'tokens': ['FOR','IF','FUNC','CLASS','IMPORT','RETURN','ASSIGN','CALL','COMMENT','DECOR'],
        'T': 0.8, 'k': 30, 'p': 1.0, 'color': '#ffd93d',
        'description': '💻 Many valid continuations
Need: diverse + guarded',
    },
    'Creative': {
        'logits': np.array([1.5,1.4,1.2,1.0,0.8,0.6,0.4,0.2,0.0,-0.2]),
        'tokens': ['BRAVE','QUIRKY','DARK','HUMOROUS','POIGNANT','SARCASTIC','TENDER','WHIMSICAL','TRAGIC','MYSTICAL'],
        'T': 1.1, 'k': 0, 'p': 0.85, 'color': '#74b9ff',
        'description': '✨ Broad exploration
Need: novel + coherent',
    },
}

print("✅ Setup done! 4 task profiles loaded.
")
print(f"{'Task':20s} {'T':5s} {'k':6s} {'p':5s} {'Expected entropy'}")
print("-" * 55)
for name, cfg in TASKS.items():
    final, ns = pipeline(cfg['logits'], cfg['T'], cfg['k'], cfg['p'])
    ent = -np.sum(final * np.log(final + 1e-12))
    print(f"{name:20s} {cfg['T']:<5} {cfg['k']:<6} {cfg['p']:<5} {ent:.2f} bits")


## 🎬 Graph 1 — The Task Distribution Gallery

Four tasks, four very different distributions. The shape of each bar chart tells you everything about the task.

- **Single spike** = one right answer = classification  
- **Gentle staircase** = a few good options = summarization  
- **Nearly flat plateau** = many valid options = code gen  
- **Moderate slope with tail** = diverse with some preference = creative


In [ ]:
# ── Graph 1: Task Gallery ────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('🗺️  The Task Gallery: Different Tasks Need Different Distribution Shapes',
             fontsize=14, fontweight='bold', color='#ffd93d')

for col_i, (name, cfg) in enumerate(TASKS.items()):
    baseline  = apply_temp(cfg['logits'], 1.0)
    tuned, ns = pipeline(cfg['logits'], cfg['T'], cfg['k'], cfg['p'])
    ent_base  = -np.sum(baseline * np.log(baseline + 1e-12))
    ent_tuned = -np.sum(tuned    * np.log(tuned    + 1e-12))

    # Row 0: Baseline
    ax0 = axes[0][col_i]
    ax0.bar(range(10), baseline, color='#8b949e', alpha=0.7, width=0.7)
    ax0.set_xticks(range(10)); ax0.set_xticklabels(cfg['tokens'], rotation=45, ha='right', fontsize=7)
    ax0.set_title(f'{name} — Baseline (T=1.0)
H={ent_base:.2f} bits', color='#8b949e', fontweight='bold')
    ax0.set_ylim(0, max(baseline)*1.15)

    # Row 1: Tuned
    ax1 = axes[1][col_i]
    ax1.bar(range(10), tuned, color=cfg['color'], alpha=0.85, width=0.7)
    ax1.set_xticks(range(10)); ax1.set_xticklabels(cfg['tokens'], rotation=45, ha='right', fontsize=7)
    ax1.set_title(f'T={cfg["T"]}, k={cfg["k"]}, p={cfg["p"]}
H={ent_tuned:.2f} bits, nucleus={ns}',
                 color=cfg['color'], fontweight='bold')
    ax1.set_ylim(0, max(baseline)*1.15)
    ax1.text(0.98, 0.96, cfg['description'], transform=ax1.transAxes, ha='right', va='top',
            color=cfg['color'], fontsize=8, style='italic',
            bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))

plt.tight_layout()
plt.savefig('graph1_usecases_gallery.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("\n💡 Notice how tuning AMPLIFIES the natural shape of each distribution.")
print("💡 Classification: already peaked → T=0.2 makes it a spike")
print("💡 Code Gen: already flat → T=0.8 keeps it broad (and k=30 adds a safety ceiling)")


## 🎬 Graph 2 — The Decision Flowchart (as a Map)

This graph IS the decision flowchart. Every task sits on a creativity vs precision map.

**How to read it:**
- The X-axis = "how much diversity do you want?"
- The Y-axis = "how much precision do you need?"
- The colored zones are the parameter recipes
- Find your task's dot, find which zone it's in, use that recipe


In [ ]:
# ── Graph 2: The Decision Map ────────────────────────────────────────
task_map = [
    ('Classification',  0.05, 0.99, '#ff6b9d', 'T=0.1-0.2
p=0.9
k=10-40'),
    ('Factual QA',      0.15, 0.92, '#ff9f43', 'T=0.2-0.4
p=0.9'),
    ('Code Gen',        0.40, 0.80, '#ffd93d', 'T=0.2-0.8
k=20-40'),
    ('Summarization',   0.35, 0.75, '#a8e6cf', 'T=0.5-0.6
p=0.9-0.95'),
    ('Chat',            0.55, 0.60, '#74b9ff', 'T=0.7
p=0.9
k=50'),
    ('Creative Writing',0.80, 0.35, '#a29bfe', 'T=0.9-1.1
p=0.95'),
    ('Brainstorming',   0.95, 0.15, '#fd79a8', 'T=1.2
p=1.0
k=0'),
]

fig, ax = plt.subplots(figsize=(12, 9))
fig.patch.set_facecolor('#0d1117')

# Background zones
from matplotlib.patches import FancyBboxPatch, Rectangle
zones_bg = [
    (0.0, 0.6, 0.5, 0.4, '#ff6b9d', 0.06, 'PRECISE ZONE
Low T, tight filters'),
    (0.5, 0.0, 0.5, 0.5, '#74b9ff', 0.06, 'CREATIVE ZONE
High T, loose filters'),
    (0.2, 0.2, 0.6, 0.6, '#ffd93d', 0.03, 'BALANCED ZONE
T=0.5-0.8'),
]
for (x0,y0,w,h,col,alpha,lbl) in zones_bg:
    rect = Rectangle((x0,y0),w,h,color=col,alpha=alpha,zorder=0)
    ax.add_patch(rect)
    ax.text(x0+w/2, y0+h/2, lbl, ha='center', va='center', color=col,
           alpha=0.4, fontsize=12, fontweight='bold')

# Task dots
for (name, creativity, precision, col, params) in task_map:
    ax.scatter(creativity, precision, s=300, color=col, zorder=5, edgecolors='white', lw=2)
    # Param box
    ax.annotate(f'  {name}
  {params}',
               xy=(creativity, precision), xytext=(creativity+0.02, precision-0.07),
               color=col, fontsize=8, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='#161b22', edgecolor=col, alpha=0.8))

# Diagonal "parameter path"
xs = [t[1] for t in task_map]
ys = [t[2] for t in task_map]
ax.plot(xs, ys, color='white', lw=1, ls='--', alpha=0.3, zorder=4)

ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('← Precision Needed          Diversity/Creativity Wanted →', fontsize=12)
ax.set_ylabel('← Less Accurate OK          Accuracy/Faithfulness Required →', fontsize=12)
ax.set_title('🗺️  The Task Map: Find Your Task → Find Your Parameters',
             fontsize=14, fontweight='bold', color='#ffd93d')
ax.axhline(0.5, color='#30363d', lw=1, ls='-'); ax.axvline(0.5, color='#30363d', lw=1, ls='-')

plt.tight_layout()
plt.savefig('graph2_usecases_map.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("\n💡 Diagonal rule: tasks along the top-left → bottom-right diagonal")
print("💡 Temperature is your main dial along this diagonal.")
print("💡 Top-p and top-k are fine-tuning tools AFTER you've set temperature.")


## 🎬 Graph 3 — The Calibration Process: One Knob at a Time

The right way to tune: start with T, add p, add k, add rep penalty — measuring after each step.

This graph shows how entropy changes as you layer each parameter on.


In [ ]:
# ── Graph 3: Calibration Steps ───────────────────────────────────────
# Use the code generation task as the example
logits_code = TASKS['Code Gen']['logits']

steps_cal = [
    ('Start:
T=1.0 only',       1.0,  0, 1.0, '#8b949e'),
    ('Step 1:
Add T=0.8',        0.8,  0, 1.0, '#ffd93d'),
    ('Step 2:
Add k=30',         0.8, 30, 1.0, '#ff9f43'),
    ('Step 3:
Add p=0.95',       0.8, 30, 0.95,'#a8e6cf'),
    ('Final:
All combined',      0.8, 30, 0.95,'#74b9ff'),
]

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('🔧  Calibration in Action (Code Gen task): One Change at a Time',
             fontsize=13, fontweight='bold', color='#ffd93d', y=1.02)

entropies_cal = []
for ax, (title, T, k, p, col) in zip(axes, steps_cal):
    final, ns = pipeline(logits_code, T, k, p)
    ent = -np.sum(final * np.log(final + 1e-12))
    entropies_cal.append(ent)
    ax.bar(range(10), final, color=col, alpha=0.85, width=0.7)
    ax.set_xticks(range(10))
    ax.set_xticklabels(TASKS['Code Gen']['tokens'], rotation=45, ha='right', fontsize=7)
    ax.set_ylim(0, 0.25)
    ax.set_title(f'{title}
H={ent:.2f}, n={ns}', color=col, fontweight='bold', fontsize=9)
    ax.set_ylabel('Probability' if ax==axes[0] else '')

# Entropy progression below
print("\n📈 Entropy progression during calibration:")
target_range = (2.0, 2.5)
for i, ((title, *_), ent) in enumerate(zip(steps_cal, entropies_cal)):
    in_range = '✅' if target_range[0] <= ent <= target_range[1] else '❌'
    bar = '█' * int(ent * 5)
    step_title = title.replace('
', ' ').strip()
    print(f"  {i}: {step_title:25s} H={ent:.2f} {bar}  {in_range} (target: {target_range})")

plt.tight_layout()
plt.savefig('graph3_usecases_calibration.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## 🎮 Graph 4 — The Final Challenge: Tune for Your Own Task

Design a distribution, set parameters, hit your target entropy.

**Challenge levels:**
- 🟢 Easy: Hit entropy between 1.5 and 2.0
- 🟡 Medium: Hit entropy between 1.0 and 1.2  
- 🔴 Hard: Hit entropy between 2.3 and 2.5 while keeping P(top) > 20%


In [ ]:
# ── Graph 4: Your Final Challenge ────────────────────────────────────
# 🎯 CHOOSE YOUR TARGET TASK
TASK_NAME = 'Summarization'  # ← Change to: 'Classification', 'Code Gen', 'Creative'

# 🔧 YOUR PARAMETER CHOICES
MY_T = 0.6   # ← Temperature
MY_K = 0     # ← Top-k (0=disabled)
MY_P = 0.95  # ← Top-p

# ─────────────────────────────────────────────────────────────────────
cfg = TASKS[TASK_NAME]
optimal_final, _ = pipeline(cfg['logits'], cfg['T'], cfg['k'], cfg['p'])
my_final, my_ns  = pipeline(cfg['logits'], MY_T, MY_K, MY_P)

ent_optimal = -np.sum(optimal_final * np.log(optimal_final + 1e-12))
ent_mine    = -np.sum(my_final      * np.log(my_final      + 1e-12))

target_ranges = {
    'Classification': (0.0, 0.8),
    'Summarization':  (1.5, 2.0),
    'Code Gen':       (2.0, 2.5),
    'Creative':       (2.0, 2.8),
}
lo, hi = target_ranges[TASK_NAME]
success = lo <= ent_mine <= hi

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'🎯 Final Challenge: {TASK_NAME}  |  Target entropy: {lo}–{hi} bits',
             fontsize=14, fontweight='bold', color='#ffd93d')

ax1.bar(range(10), optimal_final, color=cfg['color'], alpha=0.85, width=0.7)
ax1.set_xticks(range(10)); ax1.set_xticklabels(cfg['tokens'], rotation=45, ha='right', fontsize=7)
ax1.set_title(f'✅ Optimal Settings
T={cfg["T"]}, k={cfg["k"]}, p={cfg["p"]}
H={ent_optimal:.2f}',
             color=cfg['color'], fontweight='bold')

ax2.bar(range(10), my_final, color='#a29bfe', alpha=0.85, width=0.7)
ax2.set_xticks(range(10)); ax2.set_xticklabels(cfg['tokens'], rotation=45, ha='right', fontsize=7)
ax2.set_title(f'🎮 YOUR Settings
T={MY_T}, k={MY_K}, p={MY_P}
H={ent_mine:.2f}',
             color='#a29bfe', fontweight='bold')

ax3.axis('off')
result = '✅ NAILED IT!' if success else '❌ Try again!'
result_col = '#a8e6cf' if success else '#ff6b9d'
feedback  = f'Your entropy: {ent_mine:.2f} bits
Target: {lo}–{hi} bits
Nucleus: {my_ns} tokens
Top token: {cfg["tokens"][np.argmax(my_final)]} ({my_final.max():.1%})'
diff = ent_mine - ent_optimal
hint = '→ Lower T or tighten p' if ent_mine > hi else '→ Raise T or loosen p' if ent_mine < lo else '→ Perfect!'
ax3.text(0.5, 0.6, result, ha='center', va='center', transform=ax3.transAxes,
        fontsize=28, color=result_col, fontweight='bold',
        bbox=dict(boxstyle='round,pad=1', facecolor='#21262d', edgecolor=result_col, lw=3))
ax3.text(0.5, 0.2, f'{feedback}

{hint}', ha='center', va='center', transform=ax3.transAxes,
        fontsize=11, color='white')

plt.tight_layout()
plt.savefig('graph4_usecases_challenge.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## ✅ Series Complete — You Did It! 🎉

### What You Can Now Do

| Skill | Status |
|---|---|
| Explain temperature in plain English | ✅ |
| Read and interpret any distribution graph | ✅ |
| Explain why top-p adapts but top-k doesn't | ✅ |
| Trace a token through the full pipeline | ✅ |
| Diagnose and fix repetition loops | ✅ |
| Choose parameters for any task type | ✅ |
| Use entropy as a tuning compass | ✅ |

### The 5 Rules to Never Forget

```
1. Temperature first. It's the master dial.
2. Top-p adapts to confidence. Top-k doesn't.
3. Pipeline order: Temperature → Top-k → Top-p → Sample
4. Repetition penalty is a discouragement, not a ban.
5. Match your distribution to your task, not the other way around.
```

### Your Cheat Sheet

| Task | T | p | k | rep |
|------|---|---|---|-----|
| Classification | 0.1–0.2 | 0.9 | 10–40 | 1.0 |
| Factual QA | 0.2–0.4 | 0.9 | 20 | 1.0 |
| Code Gen | 0.2–0.8 | 0.95 | 20–40 | 1.0–1.1 |
| Summarization | 0.5–0.6 | 0.9–0.95 | 50 | 1.0–1.1 |
| Chat | 0.7 | 0.9 | 50 | 1.1–1.2 |
| Creative | 0.9–1.1 | 0.95 | 100 | 1.2 |
| Brainstorm | 1.2 | 1.0 | off | 1.3 |

> **🎓 You are no longer a beginner. You understand what's happening inside the model when it generates text — and that's more than most engineers know.**
